# Lab 2 – Aerial House Segmentation
**CEG4195/SEG4180 · uOttawa Winter 2026**

This notebook walks through the full pipeline:
1. Secrets injection via `.env`
2. Dataset download & preparation
3. Pixel mask generation (Week 7 code)
4. UNet training with transfer learning
5. Evaluation (IoU, Dice) on the test set
6. Visualisation of predictions

In [ ]:
# ── 0. Install dependencies ───────────────────────────────────────────────────
# (Run this cell once if not already installed)
# !pip install -r ../requirements.txt

## 1. Secrets Injection
Sensitive values (HuggingFace token, model path) are loaded from `.env` via
`python-dotenv` — never hard-coded in source.

In [ ]:
import os
from dotenv import load_dotenv

# Loads .env from the project root (one level up from notebooks/)
load_dotenv(dotenv_path="../.env")

HF_TOKEN   = os.getenv("HF_TOKEN",   "")
MODEL_PATH = os.getenv("MODEL_PATH", "../checkpoints/best_model.pth")
IMAGE_SIZE = int(os.getenv("IMAGE_SIZE", "256"))

print(f"HF_TOKEN set : {'yes' if HF_TOKEN else 'no (public datasets only)'}")
print(f"MODEL_PATH   : {MODEL_PATH}")
print(f"IMAGE_SIZE   : {IMAGE_SIZE}")

## 2. Dataset Preparation

In [ ]:
import sys
sys.path.insert(0, "..")

from dataset.prepare_dataset import prepare

prepare(
    output_dir   = "../data",
    train_frac   = 0.70,
    val_frac     = 0.15,
    seed         = 42,
    dataset_name = "keremberke/aerial-building-segmentation",
)

## 3. Pixel Mask Generation (Week 7)
Demonstrates the colour-threshold approach on a sample aerial image.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from dataset.pixel_mask_generation import generate_mask

# Pick the first training image as a demo
sample_path = sorted(Path("../data/train/images").glob("*.png"))[0]
generated_mask = generate_mask(str(sample_path), method="threshold")

img_bgr = cv2.imread(str(sample_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_rgb);         axes[0].set_title("Aerial Image");         axes[0].axis("off")
axes[1].imshow(generated_mask, cmap="gray"); axes[1].set_title("Generated Mask (threshold)"); axes[1].axis("off")
plt.suptitle("Week 7 – Pixel Mask Generation")
plt.tight_layout()
plt.show()

## 4. Data Augmentation Preview

In [ ]:
from torch.utils.data import DataLoader
from model.train import AerialDataset

train_ds = AerialDataset("../data/train", img_size=IMAGE_SIZE, augment=True)
loader   = DataLoader(train_ds, batch_size=4, shuffle=True)

imgs, masks = next(iter(loader))   # (4, 3, H, W), (4, 1, H, W)

_MEAN = np.array([0.485, 0.456, 0.406])
_STD  = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i in range(4):
    img_np = (imgs[i].permute(1,2,0).numpy() * _STD + _MEAN).clip(0,1)
    axes[0, i].imshow(img_np);                         axes[0, i].axis("off")
    axes[1, i].imshow(masks[i,0].numpy(), cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_ylabel("Image", fontsize=12)
axes[1, 0].set_ylabel("Mask",  fontsize=12)
plt.suptitle("Augmented Training Samples")
plt.tight_layout()
plt.show()

print(f"Train: {len(train_ds)}  |  Image shape: {tuple(imgs.shape)}  |  Mask shape: {tuple(masks.shape)}")

## 5. Model Architecture

In [ ]:
import torch
from model.unet import build_transfer_unet

model  = build_transfer_unet(encoder="resnet34", pretrained=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {n_params:,}")
print(f"Trainable params : {n_train:,}")
print(f"Device           : {device}")

# Quick forward pass check
dummy  = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
output = model(dummy)
print(f"Input  shape : {tuple(dummy.shape)}")
print(f"Output shape : {tuple(output.shape)}   ← raw logits")

## 6. Training

In [ ]:
# ── Run the training script as a subprocess so stdout is streamed here ────────
import subprocess, sys
result = subprocess.run(
    [
        sys.executable, "../model/train.py",
        "--data_dir",   "../data",
        "--ckpt_dir",   "../checkpoints",
        "--epochs",     "30",
        "--batch_size", "8",
        "--lr",         "1e-4",
        "--img_size",   str(IMAGE_SIZE),
    ],
    capture_output=False,
)
print(f"Exit code: {result.returncode}")

### Training curves

In [ ]:
from IPython.display import Image as IPImage
IPImage("../checkpoints/training_curves.png", width=800)

## 7. Evaluation on Test Set

In [ ]:
result = subprocess.run(
    [
        sys.executable, "../model/evaluate.py",
        "--data_dir",   "../data",
        "--ckpt_path",  "../checkpoints/best_model.pth",
        "--img_size",   str(IMAGE_SIZE),
        "--batch_size", "8",
        "--out_dir",    "../results",
    ],
    capture_output=False,
)
print(f"Exit code: {result.returncode}")

In [ ]:
IPImage("../results/metric_distributions.png", width=700)

In [ ]:
IPImage("../results/predictions_batch00.png", width=900)

## 8. Live API Inference Demo
Test the Flask API with a real aerial image.

In [ ]:
import base64, io, requests
from PIL import Image as PILImage

API_URL = "http://localhost:5001"   # docker-compose maps host:5001 → container:5000

# Health check
resp = requests.get(f"{API_URL}/")
print(resp.json())

In [ ]:
# Send a test image to /segment
test_img_path = sorted(Path("../data/test/images").glob("*.png"))[0]

with open(test_img_path, "rb") as f:
    resp = requests.post(
        f"{API_URL}/segment",
        files={"file": ("test.png", f, "image/png")},
    )

print(f"Status : {resp.status_code}")
data = resp.json()
print(f"Metrics: {data.get('metrics')}")

# Decode and display the returned mask
mask_bytes = base64.b64decode(data["mask_b64"])
pred_mask  = PILImage.open(io.BytesIO(mask_bytes))

orig_img = PILImage.open(test_img_path).convert("RGB")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(orig_img);        axes[0].set_title("Input Image");       axes[0].axis("off")
axes[1].imshow(pred_mask, cmap="gray"); axes[1].set_title("Predicted Mask"); axes[1].axis("off")
plt.tight_layout()
plt.show()